# Pulizia dei dati
Trasforma la tabella dei problemi dell'esplorazione in regole che si applicano da sole.\
Ogni regola fa due cose: corregge il dato e annota nel resoconto quante righe ha toccato e come.\
Il resoconto è lo strato di validazione: l'agente non nasconde cosa ha corretto, lo dichiara.\
Regola generale: si corregge solo ciò che si può ricostruire con certezza da un'altra colonna; il resto si tiene e si segnala.

## 1. Caricamento
Stessi file dell'esplorazione, letti allo stesso modo.\
16569 righe di vendite e 4514 di ordini: è il punto di partenza del resoconto.

In [1]:
import pandas as pd
vendite = pd.read_csv("../data/vendite.csv")
ordini = pd.read_csv("../data/ordini.csv")
print(vendite.shape, ordini.shape)

(16569, 20) (4514, 16)


## 2. Costanti
INIZIO e FINE delimitano il periodo dei dati: una data fuori da qui è sbagliata.\
NOMI_TAGLIE traduce le taglie scritte per esteso nella sigla.\
COLORI_SINONIMI riporta all'elenco chiuso i colori scritti in modo diverso: "blu navy" è un blu.

In [3]:
INIZIO, FINE = pd.Timestamp("2023-03-02"), pd.Timestamp("2026-03-02")
NOMI_TAGLIE = {"SMALL": "S", "MEDIUM": "M", "LARGE": "L", "EXTRA LARGE": "XL"}
COLORI_SINONIMI = {"blu navy": "blu"}

## 3. La forma giusta di un nome
forma_canonica() serve per i brand. Toglie gli spazi, mette tutto in minuscolo per raggruppare le varianti, poi sceglie per ogni gruppo la scrittura più frequente.\
"LARDONI", "lardoni" e "Lardoni " finiscono nello stesso gruppo e diventano "Lardoni", perché è la forma che compare 2308 volte.\
Così non serve scrivere a mano l'elenco dei brand: lo deduce dai dati.

In [5]:
def forma_canonica(colonna):
    pulita = colonna.str.strip()
    chiave = pulita.str.lower()
    canonica = pulita.groupby(chiave).agg(lambda s: s.value_counts().index[0])
    return chiave.map(canonica)

## 4. Pulizia delle vendite
nota() aggiunge una riga al resoconto. Le regole seguono un ordine preciso, perché alcune dipendono dalle precedenti.\
Duplicati: tolti per primi, altrimenti ogni conteggio successivo conterebbe due volte la stessa riga.\
Date: si convertono i due formati separatamente, ognuno con il suo schema esatto. Le date fuori periodo prendono l'anno dallo scontrino (S2023-... vuol dire 2023), perché lo scontrino lo scrive la cassa.\
Taglie, brand, colori: spazi tolti e maiuscole uniformate.\
Prezzi: prima la virgola, poi il segno. Il prezzo è il dato più affidabile, perché è il denaro incassato.\
Sconti: si ricalcolano dal prezzo quando non tornano. Per questo vengono dopo i prezzi.\
Prezzi mancanti: si ricostruiscono da listino e sconto. Per questo vengono dopo gli sconti.\
Motivo mancante: 0 è "nessuno", fino a 10 è "cliente_abituale", oltre è "saldo", perché i saldi partono dal 30%.

In [7]:
def pulisci_vendite(vendite):
    v = vendite.copy()
    resoconto = []

    def nota(regola, righe, azione):
        resoconto.append({"file": "vendite", "regola": regola, "righe": int(righe), "azione": azione})

    n = len(v)
    v = v.drop_duplicates()
    nota("righe duplicate", n - len(v), "eliminate")

    italiane = v["data_ora"].str.contains("/")
    iso = pd.to_datetime(v["data_ora"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
    ita = pd.to_datetime(v["data_ora"], format="%d/%m/%Y %H:%M", errors="coerce")
    v["data_ora"] = iso.fillna(ita)
    nota("date in formato italiano", italiane.sum(), "convertite")

    fuori = (v["data_ora"] < INIZIO) | (v["data_ora"] >= FINE)
    anno = v.loc[fuori, "scontrino"].str[1:5].astype(int)
    v.loc[fuori, "data_ora"] = [d.replace(year=a) for d, a in zip(v.loc[fuori, "data_ora"], anno)]
    nota("date fuori periodo", fuori.sum(), "anno corretto con quello dello scontrino")

    prima = v["taglia"].copy()
    t = v["taglia"].str.strip().str.upper().str.removeprefix("TG ")
    v["taglia"] = t.replace(NOMI_TAGLIE)
    nota("taglie scritte in modo diverso", (prima.notna() & (prima != v["taglia"])).sum(), "uniformate")

    prima = v["brand"].copy()
    v["brand"] = forma_canonica(v["brand"])
    nota("brand scritti in modo diverso", (prima.notna() & (prima != v["brand"])).sum(), "uniformati")

    prima = v["colore"].copy()
    v["colore"] = v["colore"].str.strip().str.lower().replace(COLORI_SINONIMI)
    nota("colori scritti in modo diverso", (prima.notna() & (prima != v["colore"])).sum(), "uniformati")

    virgola = v["prezzo_venduto"].str.contains(",", na=False)
    v["prezzo_venduto"] = pd.to_numeric(v["prezzo_venduto"].str.replace(",", "."))
    nota("prezzi con la virgola", virgola.sum(), "convertiti in numero")

    negativi = v["prezzo_venduto"] < 0
    v.loc[negativi, "prezzo_venduto"] = -v.loc[negativi, "prezzo_venduto"]
    nota("prezzi negativi", negativi.sum(), "segno corretto")

    fuori_range = ~v["sconto_pct"].between(0, 100)
    calcolato = ((1 - v["prezzo_venduto"] / v["prezzo_listino"]) * 100).round()
    sbagliato = calcolato.notna() & ((calcolato - v["sconto_pct"]).abs() > 1)
    v.loc[sbagliato, "sconto_pct"] = calcolato[sbagliato].astype(int)
    nota("sconti fuori da 0-100", (fuori_range & sbagliato).sum(), "ricalcolati dai prezzi")
    nota("sconti che non tornano con i prezzi", (~fuori_range & sbagliato).sum(), "ricalcolati dai prezzi")

    mancante = v["prezzo_venduto"].isna()
    v.loc[mancante, "prezzo_venduto"] = (v.loc[mancante, "prezzo_listino"]
                                         * (1 - v.loc[mancante, "sconto_pct"] / 100)).round(2)
    nota("prezzi mancanti", mancante.sum(), "ricostruiti da listino e sconto")

    mancante = v["motivo_sconto"].isna()
    s = v.loc[mancante, "sconto_pct"]
    v.loc[mancante, "motivo_sconto"] = s.map(lambda x: "nessuno" if x == 0
                                             else "cliente_abituale" if x <= 10 else "saldo")
    nota("motivo dello sconto mancante", mancante.sum(), "dedotto dallo sconto")

    return v.reset_index(drop=True), resoconto

## 5. Pulizia degli ordini
Date invertite: si scambiano, perché la merce non può arrivare prima dell'ordine.\
Ricevuti oltre gli ordinati: si portano agli ordinati, perché il fornitore non manda più di quanto chiesto.\
Listino diverso: vince quello delle vendite, perché è il prezzo battuto in cassa.\
Costo oltre il listino: si prende il costo delle altre righe dello stesso articolo, perché un articolo ha un solo costo.\
Il listino si corregge prima del costo, altrimenti il confronto costo-listino userebbe un listino sbagliato.

In [8]:
def pulisci_ordini(ordini, listino_vendite):
    o = ordini.copy()
    resoconto = []

    def nota(regola, righe, azione):
        resoconto.append({"file": "ordini", "regola": regola, "righe": int(righe), "azione": azione})

    n = len(o)
    o = o.drop_duplicates()
    nota("righe duplicate", n - len(o), "eliminate")

    o["data_ordine"] = pd.to_datetime(o["data_ordine"])
    o["data_arrivo"] = pd.to_datetime(o["data_arrivo"])
    invertite = o["data_arrivo"] < o["data_ordine"]
    o.loc[invertite, ["data_ordine", "data_arrivo"]] = o.loc[invertite, ["data_arrivo", "data_ordine"]].to_numpy()
    nota("arrivo prima dell'ordine", invertite.sum(), "date scambiate")

    oltre = o["qta_ricevuta"] > o["qta_ordinata"]
    o.loc[oltre, "qta_ricevuta"] = o.loc[oltre, "qta_ordinata"]
    nota("ricevuti più degli ordinati", oltre.sum(), "ricevuti portati agli ordinati")

    atteso = o["articolo"].map(listino_vendite)
    diverso = atteso.notna() & (o["prezzo_listino"] != atteso)
    o.loc[diverso, "prezzo_listino"] = atteso[diverso]
    nota("listino diverso da quello delle vendite", diverso.sum(), "preso il listino delle vendite")

    oltre = o["costo_unitario"] > o["prezzo_listino"]
    costo_tipico = o[~oltre].groupby("articolo")["costo_unitario"].agg(lambda s: s.mode()[0])
    o.loc[oltre, "costo_unitario"] = o.loc[oltre, "articolo"].map(costo_tipico)
    nota("costo più alto del listino", oltre.sum(), "preso il costo delle altre righe dello stesso articolo")

    return o.reset_index(drop=True), resoconto

## 6. La funzione che fa tutto
pulisci() chiama le due funzioni, poi sistema ciò che riguarda entrambi i file: gli articoli venduti ma mai ordinati.\
Per ogni codice orfano cerca negli ordini un articolo con stesso brand, categoria, colore, stagione e listino. Se ne trova uno solo, il codice è quello. Se ne trova più di uno, non sceglie a caso: tiene la riga e la segnala.\
Il resoconto finale mostra solo le regole che hanno toccato almeno una riga.\
È l'unica funzione che l'agente chiamerà.

In [9]:
def pulisci(vendite, ordini):
    v, r1 = pulisci_vendite(vendite)
    listino_vendite = v.groupby("articolo")["prezzo_listino"].agg(lambda s: s.mode()[0])
    o, r2 = pulisci_ordini(ordini, listino_vendite)

    chiave = ["brand", "categoria", "colore", "stagione", "prezzo_listino"]
    anagrafica = o.drop_duplicates("articolo")[["articolo"] + chiave]
    unici = anagrafica[~anagrafica.duplicated(chiave, keep=False)].set_index(chiave)["articolo"]
    orfani = ~v["articolo"].isin(o["articolo"]) & v["articolo"].notna()
    trovato = pd.Series(list(zip(*[v.loc[orfani, c] for c in chiave])), index=v.index[orfani]).map(unici)
    v.loc[trovato.dropna().index, "articolo"] = trovato.dropna()
    r1.append({"file": "vendite", "regola": "articoli mai ordinati", "righe": int(trovato.notna().sum()),
               "azione": "codice ritrovato negli ordini con brand, categoria, colore, stagione e listino"})
    r1.append({"file": "vendite", "regola": "articoli mai ordinati", "righe": int(trovato.isna().sum()),
               "azione": "tenuti e segnalati: codice non ritrovabile, costo e margine non calcolabili"})

    resoconto = pd.DataFrame(r1 + r2)
    resoconto = resoconto[resoconto["righe"] > 0].reset_index(drop=True)
    return v, o, resoconto

## 7. Esecuzione e resoconto
Applica la pulizia e mostra il resoconto.\
Le vendite passano da 16569 a 16405 righe: le 164 tolte sono i duplicati, nient'altro.\
Le date in formato italiano sono 492 e non 498, perché 6 stavano sulle righe duplicate.\
7 codici orfani su 8 vengono ritrovati, l'ultimo ha due candidati e resta segnalato.

In [10]:
vendite_pulite, ordini_puliti, resoconto = pulisci(vendite, ordini)
print(vendite_pulite.shape, ordini_puliti.shape)
resoconto

(16405, 20) (4511, 16)


,file,regola,righe,azione
0,vendite,righe duplicate,164,eliminate
1,vendite,date in formato italiano,492,convertite
2,vendite,date fuori periodo,3,anno corretto con quello dello scontrino
3,vendite,taglie scritte in modo diverso,385,uniformate
4,vendite,brand scritti in modo diverso,328,uniformati
5,vendite,colori scritti in modo diverso,164,uniformati
6,vendite,prezzi con la virgola,328,convertiti in numero
7,vendite,prezzi negativi,8,segno corretto
8,vendite,sconti fuori da 0-100,5,ricalcolati dai prezzi
9,vendite,sconti che non tornano con i prezzi,82,ricalcolati dai prezzi


## 8. Tipi di dato dopo la pulizia
data_ora ora è una data e prezzo_venduto un numero: si può calcolare.

In [11]:
vendite_pulite.info()

<class 'pandas.DataFrame'>
RangeIndex: 16405 entries, 0 to 16404
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id_riga           16405 non-null  int64         
 1   data_ora          16405 non-null  datetime64[us]
 2   scontrino         16405 non-null  str           
 3   articolo          16313 non-null  str           
 4   brand             16313 non-null  str           
 5   categoria         16313 non-null  str           
 6   taglia            16313 non-null  str           
 7   colore            16313 non-null  str           
 8   stagione          16313 non-null  str           
 9   prezzo_listino    16405 non-null  float64       
 10  prezzo_venduto    16405 non-null  float64       
 11  sconto_pct        16405 non-null  int64         
 12  motivo_sconto     16405 non-null  str           
 13  tipo_movimento    16405 non-null  str           
 14  rif_vendita       479 non-null   

## 9. Verifica contro il riferimento
Confronta, cella per cella, i dati puliti con quelli di valutazione/, cioè i dati veri prima degli errori di scrittura.\
Questo controllo si può fare solo qui, perché i dati li abbiamo generati noi. L'agente non vede mai valutazione/.\
Resta una sola cella diversa: il codice orfano con due candidati, che abbiamo scelto di non indovinare.

In [12]:
rif_v = pd.read_csv("../valutazione/vendite_riferimento.csv", parse_dates=["data_ora"])
rif_o = pd.read_csv("../valutazione/ordini_riferimento.csv", parse_dates=["data_ordine", "data_arrivo"])


def differenze(pulito, riferimento, chiave):
    a = pulito.set_index(chiave).sort_index()
    b = riferimento.set_index(chiave).sort_index()
    conta = {}
    for col in b.columns:
        uguali = (a[col] == b[col]) | (a[col].isna() & b[col].isna())
        if a[col].dtype.kind == "f":
            uguali = uguali | ((a[col] - b[col]).abs() < 0.011)
        conta[col] = int((~uguali).sum())
    return pd.Series(conta)[lambda s: s > 0]


print("Righe vendite:", len(vendite_pulite), "contro", len(rif_v))
print("Righe ordini:", len(ordini_puliti), "contro", len(rif_o))
print("Celle diverse nelle vendite:")
print(differenze(vendite_pulite, rif_v, "id_riga"))
print("Celle diverse negli ordini:")
print(differenze(ordini_puliti, rif_o, "id_ordine"))

Righe vendite: 16405 contro 16405
Righe ordini: 4511 contro 4511
Celle diverse nelle vendite:
articolo    1
dtype: int64
Celle diverse negli ordini:
Series([], dtype: int64)
